In [1]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

# Add highlighting specifications
pp.add_highlight(which='tags', style='gray')
pp.add_highlight(region='cre', style='lightskyblue')
pp.add_highlight(which='lower gap', style='bold yellow')

# Step 1: Create template Pool
template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')
template_pool.print_library();

template_pool: seq_length=51, num_values=1
state  seq
    0  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA



In [2]:
# Step 2: Randomly mutate <cre> region at 20% per nucleotide
# Step 3: Repeat each mutant 3 times
# Step 4: Insert random barcodes at <bc/>
mutagenized_bc_pool = template_pool\
    .mutagenize(region='cre', mutation_rate=0.2, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')\
    .repeat_states(3, seq_name_prefix='v', op_iter_order=-1)\
    .insert_kmers('bc', length=5).named('mutagenized_bc_pool')
mutagenized_bc_pool.print_library();

mutagenized_bc_pool: seq_length=56, num_values=15
state  name      seq
    0  mut_0.v0  TCCCGACT<cre>GtAAAtgGGGCAcTGAGCAatCAGaAcA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1  TCCCGACT<cre>GtAAAtgGGGCAcTGAGCAatCAGaAcA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_0.v2  TCCCGACT<cre>GtAAAtgGGGCAcTGAGCAatCAGaAcA</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v0  TCCCGACT<cre>GGAAAtCGGcCAtTGAGtACAtgtaAAA</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_1.v1  TCCCGACT<cre>GGAAAtCGGcCAtTGAGtACAtgtaAAA</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_1.v2  TCCCGACT<cre>GGAAAtCGGcCAtTGAGtACAtgtaAAA</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_2.v0  TCCCGACT<cre>GGAccGgGGGCgtTGAGCACgtAcGAAA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_2.v1  TCCCGACT<cre>GGAccGgGGGCgtTGAGCACgtAcGAAA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_2.v2  TCCCGACT<cre>GGAccGgGGGCgtTGAGCACgtAcGAAA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_3.v0  TCCCGACT<cre>GGAAcGgGGGCAGTatGCACACtacAgA</cre>ATTACGG<bc>gggcg</b

In [3]:
pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='hybrid', 
                                        num_hybrid_states=5, 
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        seq_names = ['site_a', 'site_b'],
                        mode='sequential', 
                        op_iter_order=-1).named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential', 
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool])\
    .repeat_states(2, seq_name_prefix='v', op_iter_order=-2)\
    .insert_kmers('bc', length=5)\
    .named('combo_pool')\

combo_pool.print_library()

combo_pool: seq_length=None, num_values=40
state  name             seq
    0  mut_0.v0         TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1         TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_1.v0         TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v1         TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_2.v0         TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_2.v1         TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_3.v0         TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_3.v1         TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_4.v0         TCCCGACT<cre>GGgAAGCGGGCtGTGAGCACACAGGAAA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_4.v1     

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_values=40)

In [ ]:
combo_pool.print_dag()

In [ ]:
df = combo_pool.generate_library(report_design_cards=True)
df.columns

In [ ]:
renamed_cols = {
    'name': 'name',
    'op[10]:get_kmers.key.kmer': 'bc_seq',
    'op[9]:repeat.value': 'v',
    'op[1]:mutagenize.key.positions': 'mut_positions',
    'op[1]:mutagenize.key.wt_chars': 'mut_wt_chars',
    'op[1]:mutagenize.key.mut_chars': 'mut_mut_chars',
    'op[2]:deletion_scan(marker_scan).key.start': 'del_start',
    'op[2]:deletion_scan(marker_scan).key.stop': 'del_stop',
    #'op[2]:deletion_scan(marker_scan).key.length': 'del_length',
    'op[2]:deletion_scan(marker_scan).key.region_content': 'deleted_seq',
    'op[6]:insertion_scan(marker_scan).key.start': 'ins_start',
    'op[6]:insertion_scan(marker_scan).key.stop': 'ins_stop',
    #'op[6]:insertion_scan(marker_scan).key.length': 'ins_length',
    'op[6]:insertion_scan(marker_scan).key.strand': 'ins_strand',
    'op[6]:insertion_scan(marker_scan).key.region_content': 'replaced_seq',
    'op[4]:from_seqs.key.seq_name': 'site_name',
    'sites_pool.seq': 'site_seq',
}
tmp_df = df.rename(columns=renamed_cols)[renamed_cols.values()]
display(tmp_df)

In [ ]:
df = combo_pool.clear_annotation()\
    .generate_library(report_design_cards=False)
df.to_excel('~/Desktop/library.xlsx', index=False)

In [ ]:
combo_pool.print_dag()

In [ ]:
combo_pool.counter.print_dag()

In [ ]:
counter_df = combo_pool.counter.get_iteration_df()
counter_df.to_excel('~/Desktop/counter_states.xlsx')

In [ ]:
import statetracker as st
A = st.State(3, name='A')
B = st.State(4, name='B')
C = st.product([A,B], name='C')
C.get_iteration_df().T


In [ ]:
import statetracker as st
A = st.State(3, name='A')
B = st.State(4, name='B')
C = st.stack([A,B], name='C')
C.get_iteration_df().T

In [ ]:
import statetracker as st
A = st.State(2,name='A')
C = st.repeat(A,3,name='C')
C.get_iteration_df().T

In [ ]:
import statetracker as st
A = st.State(12,name='A')
C = st.slice(A,None,None,3,name='C')
C.get_iteration_df().T